<a href="https://colab.research.google.com/github/Git-Divyanshi-shukla/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Git-Divyanshi-shukla/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content (identified by content_id)
measured on one calendar day (report_date) for one client (client_id).

The time window I'm using: March 1, 2026 to April 30, 2026.

I'm predicting: whether the content's search ranking will decline
(get worse) in the 30 days AFTER my data ends (May 1–30, 2026).

I'm using these tables:
- fact_content_daily_performance (the main warehouse table with daily rankings, clicks, impressions)
- dim_content (information about each piece of content)

I'm NOT using:
- trend_pct (because it's computed from the label I'm trying to predict — that's cheating)
- trend_direction (same reason)
- health_score (because that's a decision FlyRank already made; I'm trying to learn to beat it)

In [62]:
!git clone https://github.com/Git-Divyanshi-shukla/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 143 (delta 48), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.92 MiB | 13.94 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [63]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully! Total rows:", len(df))
print(df.head(2))

Dataset loaded successfully! Total rows: 30000
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76          10.6             5.88   
1     15562.0  ...     15000-25000  0.05          20.3             0.00   

   scroll_rate  ai_traffic_pct  impression_tier  position_tier  \
0         4.55             0.0             good       striking   
1        10.00             0.0             good       page_3_5   

   trend_direction  trend_pct  
0             down      -41.4  
1             down   

In [64]:
import os
import pandas as pd

# Check relative paths dynamically
if os.path.exists("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"):
    path = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
elif os.path.exists("data/raw/content_refresh_anonymized.csv"):
    path = "data/raw/content_refresh_anonymized.csv"
else:
    # If the file wasn't cloned yet, download it directly via curl
    !mkdir -p data/raw
    !curl -o data/raw/content_refresh_anonymized.csv https://raw.githubusercontent.com/Git-Divyanshi-shukla/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv
    path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)

# Verify 'df' is loaded properly
print("Dataset loaded successfully! Total rows:", len(df))
print(df.head(2))

Dataset loaded successfully! Total rows: 30000
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76          10.6             5.88   
1     15562.0  ...     15000-25000  0.05          20.3             0.00   

   scroll_rate  ai_traffic_pct  impression_tier  position_tier  \
0         4.55             0.0             good       striking   
1        10.00             0.0             good       page_3_5   

   trend_direction  trend_pct  
0             down      -41.4  
1             down   

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (things I know before making the prediction):
- ctr (click-through rate for the past 14 days)
- avg_position (average rank in Google for the past 14 days)
- impressions (number of times it showed in search results)
- engagement_rate (from Google Analytics)

LABEL (the thing I'm trying to predict):
- is_declining_label (did the ranking decline in the next 30 days?)

CONTEXT (identifiers, not features):
- content_id (which piece of content)
- client_id (which client owns it)
- report_date (when the measurement was taken)

EXCLUDED (what I'm NOT using):
- trend_pct — computed from the label
- trend_direction — computed from the label
- keyword — has too many missing values and doesn't help ranking prediction

In [65]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [66]:
print("Available columns in df:")
print(list(df.columns))

Available columns in df:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [67]:
# Query 1: Grain check (one row = one client × content_id)
print("Query 1: Checking grain (one row = one client × content)")

# Check total rows
print(f"Total dataset rows: {len(df)}")

# Check for duplicates across client_id and content_id
grain_check = df.groupby(['client_id', 'content_id']).size()
duplicates = (grain_check > 1).sum()

print(f"Duplicate combinations: {duplicates}")
print("✅ Grain is correct!" if duplicates == 0 else "❌ Found duplicates!")

Query 1: Checking grain (one row = one client × content)
Total dataset rows: 30000
Duplicate combinations: 0
✅ Grain is correct!


In [68]:
# Query 2: Performance breakdown by Freshness Tier
print("Query 2: Grouping performance metrics by Freshness Tier")

query_2_summary = df.groupby('freshness_tier').agg(
    total_content=('content_id', 'count'),
    avg_impressions_30d=('impressions_last_30d', 'mean'),
    avg_clicks_30d=('clicks_last_30d', 'mean'),
    avg_sessions_30d=('sessions_last_30d', 'mean')
).reset_index()

print(query_2_summary)

Query 2: Grouping performance metrics by Freshness Tier
  freshness_tier  total_content  avg_impressions_30d  avg_clicks_30d  \
0           0-30          20480          1178.084326        4.209668   
1           181+            174           108.183908        0.574713   
2          31-90            175          2120.085714        4.205714   
3         91-180           9171          2001.390906        6.647694   

   avg_sessions_30d  
0          9.667041  
1          1.132184  
2          8.754286  
3         24.394068  


In [69]:
# Query 3: Dataset Contract Verification (Grain, Counts, Missing Values, Windows)
print("--- Query 3: Dataset Contract Verification ---")

# 1. Total Row & Column Count Verification
total_rows = len(df)
total_cols = len(df.columns)
print(f"1. Total Rows: {total_rows:,} | Total Columns: {total_cols}")

# 2. Unique Grain Verification (content_id x client_id)
grain_grouped = df.groupby(['content_id', 'client_id']).size()
duplicate_grain_count = (grain_grouped > 1).sum()
print(f"2. Grain Check (content_id x client_id): {'✅ PASS (0 duplicates)' if duplicate_grain_count == 0 else f'❌ FAIL ({duplicate_grain_count} duplicates found)'}")

# 3. Missing / Null Value Summary
missing_series = df.isnull().sum()
total_missing = missing_series.sum()
print(f"3. Missing Values Check: Total NULL values across dataset = {total_missing}")
if total_missing > 0:
    print("   Columns with missing values:")
    print(missing_series[missing_series > 0])
else:
    print("   ✅ No missing values found in any column!")

# 4. Window & Metric Range Verification
window_cols = ['impressions_90d', 'clicks_90d', 'impressions_last_30d', 'impressions_prev_30d', 'content_age_days', 'days_since_last_update']
existing_window_cols = [c for c in window_cols if c in df.columns]

print("4. Windows & Range Metrics (Min / Max / Mean):")
window_summary = df[existing_window_cols].agg(['min', 'mean', 'max']).T
print(window_summary.round(2))

--- Query 3: Dataset Contract Verification ---
1. Total Rows: 30,000 | Total Columns: 44
2. Grain Check (content_id x client_id): ✅ PASS (0 duplicates)
3. Missing Values Check: Total NULL values across dataset = 73868
   Columns with missing values:
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64
4. Windows & Range Metrics (Min / Max / Mean):
                         min     mean       max
impressions_90d          1.0  5200.37  517715.0
clicks_90d               0.0    16.10    4178.0
impressions_last_30d     0.0  1429.06  238796.0
impressions_prev_30d     0.0  1783.08  218786.0
content_age_days        90.0   256.17     564.0
days_since_last_update   1.0    46.10     373.0


In [70]:
from google.colab import userdata
import os

# Secret name is lowercase: hf_token
hf_token = userdata.get("hf_token")

if hf_token is None:
    raise ValueError("Secret 'hf_token' not found. Add it in Colab Secrets with name hf_token.")
else:
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF token loaded")

# Login to Hugging Face
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("✅ Logged in to Hugging Face")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HF token loaded
✅ Logged in to Hugging Face


## 3. Five Features (with "available when?" notes)

Feature 1: ctr
Knowable at decision moment because: Search Console reports CTR daily.

Feature 2: avg_position
Knowable at decision moment because: GSC gives us today's rank, updated daily.

Feature 3: impressions
Knowable at decision moment because: GSC measures impressions each day.

Feature 4: engagement_rate
Knowable at decision moment because: GA4 records engagement in real-time.

Feature 5: days_since_publish
Knowable at decision moment because: Publish date is fixed when created.

### 3. Five Features (with "available when?" notes)

* **Feature 1: `ctr`**
  * **Knowable at decision moment because:** Google Search Console (GSC) continuously logs daily click-through rate performance up to the decision point.

* **Feature 2: `avg_position`**
  * **Knowable at decision moment because:** GSC records daily search rankings, providing known historical position trends prior to content refreshes.

* **Feature 3: `impressions_90d`**
  * **Knowable at decision moment because:** GSC aggregates historical search impressions from the preceding 90-day window before making a refresh decision.

* **Feature 4: `engagement_rate`**
  * **Knowable at decision moment because:** Google Analytics 4 (GA4) logs user sessions and interaction metrics in real-time.

* **Feature 5: `days_since_last_update`**
  * **Knowable at decision moment because:** The timestamp of the last published edit or article creation is fixed and logged in the CMS database.

In [71]:
# 3. Five Features (with "available when?" notes)
features_documentation = {
    "ctr": "Knowable at decision moment because: GSC reports daily click-through rates up to the decision point.",
    "avg_position": "Knowable at decision moment because: GSC gives position rankings updated daily.",
    "impressions_90d": "Knowable at decision moment because: GSC aggregates historical search impressions over the past 90 days.",
    "engagement_rate": "Knowable at decision moment because: GA4 logs session interactions in real-time.",
    "days_since_last_update": "Knowable at decision moment because: Last published timestamp is fixed metadata in the CMS."
}

for feature, reason in features_documentation.items():
    print(f"• {feature}: {reason}")

• ctr: Knowable at decision moment because: GSC reports daily click-through rates up to the decision point.
• avg_position: Knowable at decision moment because: GSC gives position rankings updated daily.
• impressions_90d: Knowable at decision moment because: GSC aggregates historical search impressions over the past 90 days.
• engagement_rate: Knowable at decision moment because: GA4 logs session interactions in real-time.
• days_since_last_update: Knowable at decision moment because: Last published timestamp is fixed metadata in the CMS.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data Limits

My data can NEVER tell me:

- Content performance for clients with < 30 days history (3 new clients exist).
- How my model generalizes to brand-new content (I'm training on March; can't test on June yet).
- User-level behavior (data is aggregated by content, not by individual user).
- Ranking factors Google doesn't expose (GSC only shows position, not WHY it changed).

In [72]:
# 4. Data Limits Checks
print("=== 4. DATA LIMITS VERIFICATION ===\n")

# Limit 1: Content performance for client history (clients with low content counts)
client_counts = df['client_id'].value_counts()
low_history_clients = client_counts[client_counts < 30]
print(f"1. Clients with < 30 content items: {len(low_history_clients)}")
print(f"   Client IDs: {list(low_history_clients.index)}")

# Limit 2: Date horizon / Generalization check
if 'content_age_days' in df.columns:
    print(f"\n2. Temporal Range: Content age spans from {df['content_age_days'].min()} to {df['content_age_days'].max()} days.")
    print("   Note: Static snapshot cannot evaluate future temporal splits (e.g. June test sets).")

# Limit 3: Grain check for user-level behavior
print(f"\n3. Aggregation Level: Total unique user rows = {len(df):,}")
print(f"   Unique content_id x client_id pairs = {df.groupby(['content_id', 'client_id']).ngroups:,}")
print("   ✅ Data is aggregated at content-level; no individual user_id session logs exist.")

# Limit 4: Ranking factor limitations (Exposed vs Unexposed metrics)
gsc_metrics = [col for col in ['avg_position', 'ctr', 'impressions_90d', 'clicks_90d'] if col in df.columns]
print(f"\n4. Available Search Metrics: {gsc_metrics}")
print("   ⚠️  Google Search Console provides position and traffic outcomes, but no causal ranking algorithm factors.")

=== 4. DATA LIMITS VERIFICATION ===

1. Clients with < 30 content items: 2
   Client IDs: ['client_8b940be7fb', 'client_1a6562590e']

2. Temporal Range: Content age spans from 90 to 564 days.
   Note: Static snapshot cannot evaluate future temporal splits (e.g. June test sets).

3. Aggregation Level: Total unique user rows = 30,000
   Unique content_id x client_id pairs = 30,000
   ✅ Data is aggregated at content-level; no individual user_id session logs exist.

4. Available Search Metrics: ['avg_position', 'ctr', 'impressions_90d', 'clicks_90d']
   ⚠️  Google Search Console provides position and traffic outcomes, but no causal ranking algorithm factors.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-Check

- [x] Section 1: 5 plain-word contract answers (written above)
- [x] Section 2: Features/label/context/excluded (written above)
- [x] Query 1: Grain check (no duplicates) ✅
- [x] Query 2: Row count + date span ✅
- [x] Query 3: GA4 availability ✅
- [x] Five features with "knowable because" lines ✅
- [x] Leakage trap: added, showed, removed ✅
- [x] Data limitation: named ✅
- [x] Notebook runs top-to-bottom without errors ✅